# pyRockImageLab-01 — Core Photo Processing

Thin interactive notebook for LR box-photo extraction and HR detail mosaics.
Algorithms live in **`pyrockimagelab`** (`QuadPicker`, `stitch_chain`, `process_lr_segments`, `process_hr_segment`, …).
This notebook only sets paths / segment counts / HR photo ranges and calls the package.

**Low-res (Steps 1–2)** — pick folder + box photo → extract segments with 4-corner ortho-warp.

**High-res (Steps 3–5)** — assign detail photos → per-photo crop/ortho → stitch → mosaic ortho.

Picks persist in `_picks.json` inside the depth folder. Re-runs prefill previous corners.

Legacy reference (do not delete yet): `DMG-07_core_photo_processing.ipynb`.

## Setup

Run from the project root so `import pyrockimagelab` resolves. If dependencies are missing, uncomment the install cell, then run imports. A GUI backend (`PyQt5` / MacOSX) is required for the interactive pickers.

In [ ]:
# %pip install --quiet pillow-heif opencv-python numpy Pillow matplotlib ipywidgets PyQt5

In [ ]:
%matplotlib qt
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# Package project dir on sys.path (importable package lives under pyrockimagelab/)
_ROOT = Path(".").resolve()
_PKG_ROOT = _ROOT / "pyrockimagelab"
if str(_PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(_PKG_ROOT))

import pyrockimagelab as pril
from pyrockimagelab import (
    DEFAULT_HR_PHOTO_RANGES,
    DEFAULT_SEG_LENGTH_FT,
    N_SEGMENTS_DEFAULT,
    PICKS_FILENAME,
    QuadPicker,
    assign_hr_photos,
    list_source_images,
    load_picks,
    pick_box_photo,
    process_folder,
    process_hr_segment,
    process_hr_segments,
    process_lr_segments,
    release_working_memory,
    save_picks,
    stitch_chain,
)

print(f"pyrockimagelab {pril.__version__}  |  backend ready")

## Config

Edit **`FOLDER`**, segment count, and default HR photo index ranges here.
Paths stay in this notebook only — the package requires them as arguments (no `CoreImages` default).

In [ ]:
# ── Required path (DMG example; change per well/depth) ──
FOLDER = Path("CoreImages/42389100800000/2563-2572").resolve()
assert FOLDER.is_dir(), f"Folder not found: {FOLDER}"

# UWI = parent of the depth-range folder (required by process_* naming)
UWI = FOLDER.parent.name

# ── Method knobs (notebook overrides; package has matching defaults) ──
N_SEGMENTS = N_SEGMENTS_DEFAULT  # typically 3
HR_PHOTO_RANGES = dict(DEFAULT_HR_PHOTO_RANGES)  # {seg: (lo, hi)} half-open indices
# Example override:
# HR_PHOTO_RANGES = {1: (1, 7), 2: (7, 13), 3: (13, 19)}

RUN_LR = True
RUN_HR = True

print(f"FOLDER = {FOLDER}")
print(f"UWI    = {UWI}")
print(f"N_SEGMENTS = {N_SEGMENTS}")
print(f"HR_PHOTO_RANGES = {HR_PHOTO_RANGES}")
print(f"RUN_LR={RUN_LR}  RUN_HR={RUN_HR}")

## Step 1 — Sources, box photo, picks

Lists source images (outputs with `_lr_` / `_hr_` filtered), loads `_picks.json`, and opens the package box-photo picker.

In [ ]:
sources = list_source_images(FOLDER)
assert sources, f"No source images in {FOLDER}"

picks = load_picks(FOLDER)
picks["n_segments"] = int(N_SEGMENTS)
save_picks(FOLDER, picks)

print(f"Found {len(sources)} source image(s)")
print(f"Picks: {FOLDER / PICKS_FILENAME}  "
      f"(box quads={len(picks.get('box_quads', {}))}, "
      f"n_segments={picks.get('n_segments')})")

BOX_PATH = pick_box_photo(sources, default_path=picks.get("box_photo"))
assert BOX_PATH is not None, "No box photo selected"
picks["box_photo"] = BOX_PATH.name
save_picks(FOLDER, picks)
print(f"Active box photo: {BOX_PATH.name}")

## Step 2 — Low-res segments (`process_lr_segments`)

Qt `QuadPicker` per segment (corners prefilled when saved). Outputs go beside the UWI folder as `{UWI}_{start}-{end}.tif/.png`.

In [ ]:
if not RUN_LR:
    print("RUN_LR=False — skipping LR extraction")
else:
    saved, picks, quit_req = process_lr_segments(
        folder=FOLDER,
        box_path=BOX_PATH,
        uwi=UWI,
        folder_idx=1,
        total_folders=1,
        picks=picks,
    )
    N_SEGMENTS = int(picks.get("n_segments", N_SEGMENTS))
    print(f"\nLR segments saved this run: {saved}  |  n_segments={N_SEGMENTS}")
    if quit_req:
        print("Quit requested during LR — stop here or re-run later.")

## Step 3 — Assign HR photos (`assign_hr_photos`)

Uses notebook `HR_PHOTO_RANGES` as default ranges when no saved assignments exist (and enough photos are present). Confirm in the UI, then assignments are written to `_picks.json`.

In [ ]:
if not RUN_HR:
    print("RUN_HR=False — skipping HR assignment")
    segment_photo_indices = {
        int(k.replace("seg", "")): v
        for k, v in picks.get("hr_assignments", {}).items()
    }
    photo_names = [p.name for p in list_source_images(FOLDER)]
else:
    sources = list_source_images(FOLDER)
    photo_names = [p.name for p in sources]
    n_segments = int(picks.get("n_segments", N_SEGMENTS))

    assignments = assign_hr_photos(
        sources,
        n_segments,
        prev_assignments=picks.get("hr_assignments", {}),
        default_ranges=HR_PHOTO_RANGES,
    )
    if assignments is None:
        raise RuntimeError("HR photo assignment cancelled")

    segment_photo_indices = dict(assignments)
    picks["hr_assignments"] = {f"seg{k}": v for k, v in segment_photo_indices.items()}
    save_picks(FOLDER, picks)

    print("Confirmed photo assignments:\n")
    for seg_num, indices in sorted(segment_photo_indices.items()):
        print(f"  Segment {seg_num}: {len(indices)} photos")
        for idx in indices:
            print(f"    [{idx}] {photo_names[idx]}")

## Step 4 — Mosaic / stitch (package)

No local stitch code. `process_hr_segment` calls `stitch_chain` inside `pyrockimagelab` (SIFT + RANSAC sequential chain). Tune `NFEATURES` / `MATCH_RATIO` / etc. from the mosaic ortho picker UI when restitching.

## Step 5 — High-res segments (`process_hr_segment`)

Run the cell below to process all assigned segments, or call `process_hr_segment(...)` for one `seg_num` at a time. For a full-folder one-shot (LR+HR with package UIs), use `process_folder` in the optional cell at the bottom.

In [ ]:
if not RUN_HR:
    print("RUN_HR=False — skipping HR mosaic")
else:
    n_segments = int(picks.get("n_segments", N_SEGMENTS))
    photo_names = [p.name for p in list_source_images(FOLDER)]
    segment_photo_indices = {
        int(k.replace("seg", "")): v
        for k, v in picks.get("hr_assignments", {}).items()
    }

    hr_results = {}
    for seg_num in range(1, n_segments + 1):
        result = process_hr_segment(
            folder=FOLDER,
            uwi=UWI,
            seg_num=seg_num,
            n_segments=n_segments,
            photo_names=photo_names,
            segment_photo_indices=segment_photo_indices,
            picks=picks,
        )
        hr_results[seg_num] = result
        if result == "quit":
            print("Quit requested during HR — stop here.")
            break

    n_done = sum(1 for v in hr_results.values() if v is True)
    print(f"\nHigh-res segments done: {n_done}/{n_segments}")
    if n_done == n_segments:
        picks["hr_completed"] = True
        save_picks(FOLDER, picks)
    release_working_memory()


### Optional — one-shot `process_folder`

Runs LR + HR for this folder with package pickers (box select, LR loop, HR assign with package default ranges, then each HR segment). Prefer the stepped cells above when you need notebook `HR_PHOTO_RANGES`.

In [ ]:
# Uncomment to run the full package pipeline for FOLDER:
# lr_saved, hr_saved, quit_req = process_folder(
#     folder=FOLDER,
#     uwi=UWI,
#     folder_idx=1,
#     total_folders=1,
#     run_lr=RUN_LR,
#     run_hr=RUN_HR,
# )
# print(f"LR saved={lr_saved}  HR saved={hr_saved}  quit={quit_req}")